# Evaluation — WER & Latency Benchmarks

Reproduces the numbers cited in the README and model card:
1. **WER** of base `whisper-medium` vs the fine-tuned model on a held-out code-switched test set
2. **Latency & size** across FP32 / CT2 INT8, on CPU vs GPU

Run on Colab (GPU runtime) for the full comparison; CPU-only still works for WER + CPU latency.


In [ ]:
!pip install -q faster-whisper ctranslate2 transformers jiwer datasets librosa


In [ ]:
import time, statistics
import numpy as np
import librosa
import torch
import jiwer
from datasets import load_dataset, Audio

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

HF_FT      = 'Seif-Eldeen-Sameh/whisper-medium-arabic-codeswitched'
CT2_FT     = 'Seif-Eldeen-Sameh/whisper-medium-arabic-codeswitched-ct2'
DATASET_ID = 'Seif-Eldeen-Sameh/asr_codeswitched_dataset'


## 1. Load a held-out test slice

In [ ]:
# Use a small held-out slice for a quick, repeatable WER estimate.
ds = load_dataset(DATASET_ID, split='train')
ds = ds.cast_column('audio', Audio(sampling_rate=16000))

# Deterministic test split (same seed as training used test_size=0.01)
test = ds.train_test_split(test_size=0.01, seed=42)['test']
N = min(100, len(test))   # cap for speed; raise for a tighter estimate
test = test.select(range(N))
print(f'Evaluating on {N} examples')


## 2. Text normalization
Light normalization so WER reflects real errors, not spacing/punctuation noise. We deliberately keep Arabic + Latin characters (no MSA folding) since code-switching is the point.

In [ ]:
import re
_AR_DIAC = re.compile(r'[\u064B-\u0652]')   # tashkeel
def norm(t: str) -> str:
    t = t.lower().strip()
    t = _AR_DIAC.sub('', t)
    t = re.sub(r'[^\w\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()


## 3. WER — fine-tuned (CT2 INT8) vs base whisper-medium

In [ ]:
from faster_whisper import WhisperModel

def wer_for(model_id_or_size: str, compute='int8') -> float:
    dev = DEVICE
    ct = ('int8_float16' if dev == 'cuda' else 'int8') if compute == 'int8' else compute
    m = WhisperModel(model_id_or_size, device=dev, compute_type=ct)
    refs, hyps = [], []
    for ex in test:
        audio = np.asarray(ex['audio']['array'], dtype=np.float32)
        segs, _ = m.transcribe(audio, language='ar', beam_size=1, vad_filter=False)
        hyp = ' '.join(s.text for s in segs)
        refs.append(norm(ex['transcript']))
        hyps.append(norm(hyp))
    return 100 * jiwer.wer(refs, hyps)

# Fine-tuned CT2 build
wer_ft = wer_for(CT2_FT, compute='int8')
print(f'Fine-tuned (CT2 INT8) WER: {wer_ft:.1f}%')

# Base whisper-medium for comparison (faster-whisper pulls a CT2 build automatically)
wer_base = wer_for('medium', compute='int8')
print(f'Base whisper-medium  WER: {wer_base:.1f}%')

print(f'\nAbsolute WER reduction: {wer_base - wer_ft:.1f} points')


## 4. Latency & size — FP32 vs CT2 INT8, CPU vs GPU
Measures wall-clock transcription time on a fixed clip. Run the GPU cells on a GPU runtime.

In [ ]:
# Pick one ~30-60s clip (concatenate a few test examples if needed)
clip = np.concatenate([np.asarray(test[i]['audio']['array'], dtype=np.float32)
                       for i in range(min(8, N))])
clip_sec = len(clip) / 16000
print(f'Benchmark clip: {clip_sec:.1f}s')

def bench(model_id, compute, device, runs=3):
    m = WhisperModel(model_id, device=device, compute_type=compute)
    # warmup
    list(m.transcribe(clip, language='ar', beam_size=1)[0])
    times = []
    for _ in range(runs):
        t0 = time.time()
        list(m.transcribe(clip, language='ar', beam_size=1)[0])
        times.append(time.time() - t0)
    return statistics.median(times)

rows = []
rows.append(('CT2 INT8', 'cpu', bench(CT2_FT, 'int8', 'cpu')))
if DEVICE == 'cuda':
    rows.append(('CT2 INT8', 'cuda', bench(CT2_FT, 'int8_float16', 'cuda')))
    rows.append(('CT2 FP16', 'cuda', bench(CT2_FT, 'float16', 'cuda')))

print(f'\n{"variant":<12}{"device":<8}{"sec":>8}{"RTF":>8}')
for v, d, t in rows:
    print(f'{v:<12}{d:<8}{t:>8.2f}{t/clip_sec:>8.3f}')
print('\nRTF = real-time factor (lower is faster; <1 means faster than real time)')


## 5. Fill these into the README / model card

Copy `wer_base`, `wer_ft`, and the latency table into:
- `README.md` → Results section
- `cards/MODEL_CARD.md` → Evaluation table
